In [ ]:
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import os
import numpy as np
from PIL import Image
from base.torch_utils import distributed as dist
import time
from base.torch_utils import misc
import torch
import base.dnnlib as dnnlib
import string
from datetime import datetime
from torch.optim.lr_scheduler import ReduceLROnPlateau
import sys
sys.path.append('./base')

new_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
# new_transforms = None
val_data_dir ='/data/guidance-team-new/Imagenet/val_64/'
valid_dataset_kwargs      = dict(class_name='training.dataset.ImageFolderDataset',
         path=val_data_dir, transform=new_transforms)
data_loader_kwargs  = dict(class_name='torch.utils.data.DataLoader', pin_memory=True, num_workers=2, prefetch_factor=2)
valid_dataset_obj = dnnlib.util.construct_class_by_name(**valid_dataset_kwargs)


In [ ]:
type(valid_dataset_obj)

In [ ]:
image,label=valid_dataset_obj[0]
image.shape

In [ ]:
image,label=valid_dataset_obj[0]
image.shape

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),                # 縮放最短邊到 256
    transforms.CenterCrop(224),            # 中心裁切到 224x224
    transforms.ToTensor(),                 # 轉 Tensor (C,H,W)
    transforms.Normalize(                  # ImageNet 標準化
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])
val_data_dir ='/data/guidance-team-new/Imagenet/val_64/'
valid_dataset_kwargs      = dict(class_name='training.dataset.ImageFolderDataset',
         path=val_data_dir, transform=preprocess)
data_loader_kwargs  = dict(class_name='torch.utils.data.DataLoader', pin_memory=True, num_workers=2, prefetch_factor=2)
valid_dataset_obj = dnnlib.util.construct_class_by_name(**valid_dataset_kwargs)
data_loader_kwargs  = dict(class_name='torch.utils.data.DataLoader', pin_memory=True, num_workers=2, prefetch_factor=2)
device              = torch.device('cuda')

val_data_loader = dnnlib.util.construct_class_by_name(dataset=valid_dataset_obj,
                                                      batch_size=32,
                                                      **data_loader_kwargs)

In [ ]:
type(val_data_loader)

In [ ]:
for image,label in val_data_loader:
    print(image.shape)
    print(label.shape)
    break

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests

# ======== 1. 載入模型 ========
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.eval()

# ======== 2. 定義預處理 ========
preprocess = transforms.Compose([
    transforms.Resize(256),                # 縮放最短邊到 256
    transforms.CenterCrop(224),            # 中心裁切到 224x224
    transforms.ToTensor(),                 # 轉 Tensor (C,H,W)
    transforms.Normalize(                  # ImageNet 標準化
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])
# ======== 3. 建立資料集 ========
val_data_dir ='/data/guidance-team-new/Imagenet/val_64/'
valid_dataset_kwargs      = dict(class_name='training.dataset.ImageFolderDataset',
         path=val_data_dir, transform=preprocess)
data_loader_kwargs  = dict(class_name='torch.utils.data.DataLoader', pin_memory=True, num_workers=2, prefetch_factor=2)
valid_dataset_obj = dnnlib.util.construct_class_by_name(**valid_dataset_kwargs)
data_loader_kwargs  = dict(class_name='torch.utils.data.DataLoader', pin_memory=True, num_workers=2, prefetch_factor=2)
device              = torch.device('cuda')

val_data_loader = dnnlib.util.construct_class_by_name(dataset=valid_dataset_obj,
                                                      batch_size=32,
                                                      **data_loader_kwargs))

for image,label in val_data_loader:
    #處理所有下面步驟並計算accuracy
    #image.shape:torch.Size([32, 3, 224, 224])
    #label.shape:torch.Size([32, 1000])
# ======== 3. 讀取 PNG 圖片 ========
png_path = "/data/guidance-team-new/Imagenet/val_64/00000/img00000003.png"  # 改成你的檔案路徑
img = Image.open(png_path).convert("RGB")  # 確保是 RGB 格式

# ======== 4. 預處理 ========
img_t = preprocess(img).unsqueeze(0)  # 增加 batch 維度

# ======== 5. 預測 ========
with torch.no_grad():
    outputs = model(img_t)
    probs = torch.nn.functional.softmax(outputs[0], dim=0)

# ======== 6. 載入 ImageNet 標籤 ========
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 7. Top-5 結果 ========
top5_prob, top5_catid = torch.topk(probs, 5)
for i in range(top5_prob.size(0)):
    print(f"{labels[top5_catid[i]]}: {top5_prob[i].item():.4f}")

# ======== 8. 顯示圖片 ========
plt.imshow(img)
plt.axis("off")
plt.title(f"{labels[top5_catid[0]]} ({top5_prob[0].item():.2%})", fontsize=14, color='blue')
plt.show()


In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests

# ======== 1. 載入模型 ========
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.eval()

# ======== 2. 定義預處理 ========
preprocess = transforms.Compose([
    transforms.Resize(256),                # 縮放最短邊到 256
    transforms.CenterCrop(224),            # 中心裁切到 224x224
    transforms.ToTensor(),                 # 轉 Tensor (C,H,W)
    transforms.Normalize(                  # ImageNet 標準化
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# ======== 3. 讀取 PNG 圖片 ========
png_path = "/data/guidance-team-new/Imagenet/val_64/00000/img00000003.png"  # 改成你的檔案路徑
img = Image.open(png_path).convert("RGB")  # 確保是 RGB 格式

# ======== 4. 預處理 ========
img_t = preprocess(img).unsqueeze(0)  # 增加 batch 維度

# ======== 5. 預測 ========
with torch.no_grad():
    outputs = model(img_t)
    probs = torch.nn.functional.softmax(outputs[0], dim=0)

# ======== 6. 載入 ImageNet 標籤 ========
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 7. Top-5 結果 ========
top5_prob, top5_catid = torch.topk(probs, 5)
for i in range(top5_prob.size(0)):
    print(f"{labels[top5_catid[i]]}: {top5_prob[i].item():.4f}")

# ======== 8. 顯示圖片 ========
plt.imshow(img)
plt.axis("off")
plt.title(f"{labels[top5_catid[0]]} ({top5_prob[0].item():.2%})", fontsize=14, color='blue')
plt.show()


In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests
import os
import dnnlib
import tqdm
import sys

# ======== 1. 載入模型 ========
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.eval()

# ======== 2. 定義預處理 ========
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ======== 3. 建立資料集和 DataLoader ========
# 確保 training.dataset 模組和 dnnlib 已被正確引入
sys.path.append('./base')
import training.dataset as dataset_module

val_data_dir = '/data/guidance-team-new/Imagenet/val_64/'

# 假設 ImageFolderDataset 類別已包含在 training.dataset 模組中
valid_dataset_kwargs = dict(
    class_name='training.dataset.ImageFolderDataset',
    path=val_data_dir,
    transform=preprocess
)
valid_dataset_obj = dnnlib.util.construct_class_by_name(**valid_dataset_kwargs)

data_loader_kwargs = dict(
    class_name='torch.utils.data.DataLoader',
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_data_loader = dnnlib.util.construct_class_by_name(
    dataset=valid_dataset_obj,
    batch_size=32,
    **data_loader_kwargs
)

# ======== 4. 載入 ImageNet 標籤 ========
# 只需要載入一次標籤
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 5. 執行驗證迴圈 ========
model.to(device)
correct_predictions = 0
total_samples = 0
examples_shown = False # 新增一個旗標來確保只印出一次範例

with torch.no_grad():
    for i, (images, labels_idx) in enumerate(tqdm.tqdm(val_data_loader, total=len(val_data_loader))):
        # 將資料移到 GPU
        images = images.to(device)
        
        if labels_idx.dim() > 1:
            labels_idx = torch.argmax(labels_idx, dim=1)
        
        labels_idx = labels_idx.to(device)

        # 進行預測
        outputs = model(images)
        _, predicted_labels = torch.max(outputs, 1)

        # 計算正確預測數
        correct_predictions += (predicted_labels == labels_idx).sum().item()
        total_samples += labels_idx.size(0)

        # --- 顯示第一個批次的範例 ---
        if i == 0 and not examples_shown:
            print("\n--- 顯示第一個批次的範例預測 ---")
            
            # 從這個批次中選擇 3 張圖片來顯示
            for j in range(3):
                # 取得單張圖片、預測標籤和真實標籤
                single_image = images[j].cpu()
                single_output = outputs[j].cpu()
                true_label_idx = labels_idx[j].cpu().item()
                
                # 反正規化以顯示圖片
                # 這是預處理的反向操作
                unnormalize = transforms.Normalize(
                    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                    std=[1/0.229, 1/0.224, 1/0.225]
                )
                display_image = unnormalize(single_image)
                # 將 CHW 轉為 HWC，並將 Tensor 轉為 numpy array
                display_image = display_image.permute(1, 2, 0).numpy()
                display_image = (display_image * 255).astype('uint8') # 轉為 uint8 格式

                # 取得 Top-5 預測
                probs = torch.nn.functional.softmax(single_output, dim=0)
                top5_prob, top5_catid = torch.topk(probs, 5)

                # 顯示圖片和標題
                plt.figure()
                plt.imshow(display_image)
                plt.axis("off")
                
                title = f"True: {labels[true_label_idx]}\n"
                title += "Top-5 Predictions:\n"
                for k in range(top5_prob.size(0)):
                    title += f"  {labels[top5_catid[k]]}: {top5_prob[k].item():.2%}\n"
                
                plt.title(title, fontsize=10, color='blue')
                plt.show()

            examples_shown = True

# ======== 6. 計算並列印準確率 ========
accuracy = correct_predictions / total_samples
print(f"總共處理了 {total_samples} 張圖片。")
print(f"正確預測了 {correct_predictions} 張圖片。")
print(f"模型在驗證集上的準確率為: {accuracy:.4f}")

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests
import os
import dnnlib
import tqdm
import sys

# ======== 1. 載入模型 ========
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.eval()

# ======== 2. 定義預處理 ========
# preprocess = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])
preprocess = transforms.Compose([
    transforms.Resize(256),   # 先放大到 256x256
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(degrees=15),
    
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    
    # RandomErasing 應該放在 ToTensor 和 Normalize 之後
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.33)), 
])
# ======== 3. 建立資料集和 DataLoader ========
# 確保 training.dataset 模組和 dnnlib 已被正確引入
sys.path.append('./base')
import training.dataset as dataset_module

val_data_dir = '/data/guidance-team-new/Imagenet/val_64/'

# 假設 ImageFolderDataset 類別已包含在 training.dataset 模組中
valid_dataset_kwargs = dict(
    class_name='training.dataset.ImageFolderDataset',
    path=val_data_dir,
    transform=preprocess
)
valid_dataset_obj = dnnlib.util.construct_class_by_name(**valid_dataset_kwargs)

data_loader_kwargs = dict(
    class_name='torch.utils.data.DataLoader',
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_data_loader = dnnlib.util.construct_class_by_name(
    dataset=valid_dataset_obj,
    batch_size=32,
    **data_loader_kwargs
)

# ======== 4. 載入 ImageNet 標籤 ========
# 只需要載入一次標籤
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 5. 執行驗證迴圈 ========
model.to(device)
correct_predictions = 0
total_samples = 0
examples_shown = False # 新增一個旗標來確保只印出一次範例

with torch.no_grad():
    for i, (images, labels_idx) in enumerate(tqdm.tqdm(val_data_loader, total=len(val_data_loader))):
        # 將資料移到 GPU
        images = images.to(device)
        
        if labels_idx.dim() > 1:
            labels_idx = torch.argmax(labels_idx, dim=1)
        
        labels_idx = labels_idx.to(device)

        # 進行預測
        outputs = model(images)
        _, predicted_labels = torch.max(outputs, 1)

        # 計算正確預測數
        correct_predictions += (predicted_labels == labels_idx).sum().item()
        total_samples += labels_idx.size(0)

        # --- 顯示第一個批次的範例 ---
        if i == 0 and not examples_shown:
            print("\n--- 顯示第一個批次的範例預測 ---")
            
            # 從這個批次中選擇 3 張圖片來顯示
            for j in range(3):
                # 取得單張圖片、預測標籤和真實標籤
                single_image = images[j].cpu()
                single_output = outputs[j].cpu()
                true_label_idx = labels_idx[j].cpu().item()
                
                # 反正規化以顯示圖片
                # 這是預處理的反向操作
                unnormalize = transforms.Normalize(
                    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                    std=[1/0.229, 1/0.224, 1/0.225]
                )
                display_image = unnormalize(single_image)
                # 將 CHW 轉為 HWC，並將 Tensor 轉為 numpy array
                display_image = display_image.permute(1, 2, 0).numpy()
                display_image = (display_image * 255).astype('uint8') # 轉為 uint8 格式

                # 取得 Top-5 預測
                probs = torch.nn.functional.softmax(single_output, dim=0)
                top5_prob, top5_catid = torch.topk(probs, 5)

                # 顯示圖片和標題
                plt.figure()
                plt.imshow(display_image)
                plt.axis("off")
                
                title = f"True: {labels[true_label_idx]}\n"
                title += "Top-5 Predictions:\n"
                for k in range(top5_prob.size(0)):
                    title += f"  {labels[top5_catid[k]]}: {top5_prob[k].item():.2%}\n"
                
                plt.title(title, fontsize=10, color='blue')
                plt.show()

            examples_shown = True

# ======== 6. 計算並列印準確率 ========
accuracy = correct_predictions / total_samples
print(f"總共處理了 {total_samples} 張圖片。")
print(f"正確預測了 {correct_predictions} 張圖片。")
print(f"模型在驗證集上的準確率為: {accuracy:.4f}")